In [1]:
import numpy as np 
import pandas as pd
import joblib

df = pd.read_csv("/Users/mariahwaslie/Desktop/Coding Projects/stats-ML/data/lendingclub_model.csv")

/var/folders/zz/8h6rnwtn2s3759zf_90928740000gn/T/ipykernel_22340/350116167.py:5: DtypeWarning: Columns (0: hardship_loan_status) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/Users/mariahwaslie/Desktop/Coding Projects/stats-ML/data/lendingclub_model.csv")


In [2]:
model = joblib.load("../models/lgbm_regression_model.pkl")
feature_columns = joblib.load("../models/regression_feature_columns.pkl")
cat_cols = joblib.load("../models/cat_cols_reg.pkl")
preprocessing = joblib.load("../models/regression_feature_columns.pkl")

In [3]:
# 1. flag corrupt values (keep BEFORE overwriting)
df["dti_is_corrupt"] = (df["dti"] > 100) | (df["dti"] < 0)

# 2. replace invalid values with NaN
df.loc[df["dti"] > 100, "dti"] = np.nan
df.loc[df["dti"] < 0, "dti"] = np.nan

/var/folders/zz/8h6rnwtn2s3759zf_90928740000gn/T/ipykernel_22340/1830244924.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["dti_is_corrupt"] = (df["dti"] > 100) | (df["dti"] < 0)


In [4]:
mask = df["dti"].isna()
dti = df[mask].copy()

for col in dti.select_dtypes(include=["object"]).columns:
    dti[col] =  dti[col].astype("category")


/var/folders/zz/8h6rnwtn2s3759zf_90928740000gn/T/ipykernel_22340/3509955084.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in dti.select_dtypes(include=["object"]).columns:


In [5]:
X  = dti.drop(columns=["dti","issue_d","earliest_cr_line"])
X = X.reindex(columns=feature_columns)
print(X.shape)


(896, 142)


In [6]:
predicted_dti = model.predict(X)

In [7]:
print(df["dti"].isna().sum())

896


In [8]:
df.loc[mask, "dti"] = predicted_dti
print(df["dti"].isna().sum())

0


In [9]:
pd.set_option("display.max_info_columns", 1000)

In [10]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 1269822 entries, 0 to 1269821
Data columns (total 186 columns):
 #    Column                              Non-Null Count    Dtype  
---   ------                              --------------    -----  
 0    loan_amnt                           1269822 non-null  float64
 1    funded_amnt                         1269822 non-null  float64
 2    funded_amnt_inv                     1269822 non-null  float64
 3    term                                1269822 non-null  int64  
 4    int_rate                            1269822 non-null  float64
 5    installment                         1269822 non-null  float64
 6    grade                               1269822 non-null  str    
 7    sub_grade                           1269822 non-null  str    
 8    emp_title                           1269822 non-null  str    
 9    emp_length                          1269822 non-null  int64  
 10   home_ownership                      1269822 non-null  str    
 11   annual_

# Removing Columns that will cause data leackage

In [11]:
leakage_cols = [
    # Target / outcome
    "loan_status",

    # Remaining principal after issuance
    "out_prncp",
    "out_prncp_inv",

    # Payment history
    "total_pymnt",
    "total_pymnt_inv",
    "total_rec_prncp",
    "total_rec_int",
    "total_rec_late_fee",

    # Collections / recoveries
    "recoveries",
    "collection_recovery_fee",

    # Future payment information
    "last_pymnt_amnt",
    "last_pymnt_month",
    "last_pymnt_year",
    "last_pymnt_d",
    "next_pymnt_month",
    "next_pymnt_year",
    "next_pymnt_d",

    # Future credit information
    "last_fico_range_low",
    "last_fico_range_high",
    "last_credit_pull_month",
    "last_credit_pull_year",
    "last_credit_pull_d",

    # Hardship program information
    "hardship_flag",
    "hardship_type",
    "hardship_reason",
    "hardship_status",
    "hardship_loan_status",

    # Settlement information
    "debt_settlement_flag",
    "settlement_status",

    "hardship_start_month",
    "hardship_start_year",
    "hardship_start_date",
    "hardship_end_month",
    "hardship_end_year",
    "hardship_end_date",

    "payment_plan_start_month",
    "payment_plan_start_year",
    "payment_plan_start_date",

    "debt_settlement_flag_month",
    "debt_settlement_flag_year",
    "debt_settlement_flag_d",
    "debt_settlement_flag_date",

    "settlement_month",
    "settlement_year",
    "settlement_date"
]

In [12]:
df = df.drop(columns=leakage_cols, errors="ignore")

In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1269822 entries, 0 to 1269821
Data columns (total 147 columns):
 #    Column                              Non-Null Count    Dtype  
---   ------                              --------------    -----  
 0    loan_amnt                           1269822 non-null  float64
 1    funded_amnt                         1269822 non-null  float64
 2    funded_amnt_inv                     1269822 non-null  float64
 3    term                                1269822 non-null  int64  
 4    int_rate                            1269822 non-null  float64
 5    installment                         1269822 non-null  float64
 6    grade                               1269822 non-null  str    
 7    sub_grade                           1269822 non-null  str    
 8    emp_title                           1269822 non-null  str    
 9    emp_length                          1269822 non-null  int64  
 10   home_ownership                      1269822 non-null  str    
 11   annual_

In [14]:
date_cols = ["issue_d","earliest_cr_line","sec_app_earliest_cr_line"]

for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col].str.strip(), errors="coerce")
        print(f"{col}:", df[col])

issue_d: 0         2015-12-01
1         2015-12-01
2         2015-12-01
3         2015-12-01
4         2015-12-01
             ...    
1269817   2016-10-01
1269818   2016-10-01
1269819   2016-10-01
1269820   2016-10-01
1269821   2016-10-01
Name: issue_d, Length: 1269822, dtype: datetime64[us]
earliest_cr_line: 0         2003-08-01
1         1999-12-01
2         1998-06-01
3         1987-10-01
4         1990-06-01
             ...    
1269817   2004-07-01
1269818   2002-03-01
1269819   2011-06-01
1269820   1997-08-01
1269821   1999-07-01
Name: earliest_cr_line, Length: 1269822, dtype: datetime64[us]


/var/folders/zz/8h6rnwtn2s3759zf_90928740000gn/T/ipykernel_22340/4258645848.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col].str.strip(), errors="coerce")
/var/folders/zz/8h6rnwtn2s3759zf_90928740000gn/T/ipykernel_22340/4258645848.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col].str.strip(), errors="coerce")


In [15]:
start_year = df["issue_d"].dt.year.min()
start_month = df["issue_d"].dt.month.min()

df["issue_month_num"] = (
    (df["issue_d"].dt.year - start_year) * 12
    + (df["issue_d"].dt.month - start_month)
)
start_year_cr = df["earliest_cr_line"].dt.year.min()
start_month_cr = df["earliest_cr_line"].dt.month.min()

df["earliest_cr_line_month_num"] = (
    (df["earliest_cr_line"].dt.year - start_year_cr) * 12
    + (df["earliest_cr_line"].dt.month - start_month_cr)
)


/var/folders/zz/8h6rnwtn2s3759zf_90928740000gn/T/ipykernel_22340/1575572987.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["issue_month_num"] = (
/var/folders/zz/8h6rnwtn2s3759zf_90928740000gn/T/ipykernel_22340/1575572987.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["earliest_cr_line_month_num"] = (


In [16]:
df.drop(columns=["issue_d", "earliest_cr_line"], inplace=True)

In [17]:
print(df["issue_month_num"].value_counts())

issue_month_num
50    45217
45    42948
42    40995
47    38834
33    36018
      ...  
5      2490
80     2370
81     2097
82     1622
83     1241
Name: count, Length: 79, dtype: int64


In [18]:
df.to_csv("../data/lendingclub_model_cleaned.csv", index=False)